# 金融对话数值推理模型：SFT → 轻量 DPO 

目标：训练一个面向 **金融对话数值推理** 的 reasoning model。

整体闭环：
- **SFT-1（FinQA）**：先学习表文混合数值推理与程序监督
- **SFT-2（ConvFinQA）**：再学习多轮对话中的 follow-up 数值推理
- **DPO（可选）**：小规模优化表达质量、结构和少废话
- **GRPO（推荐）**：基于可验证 reward 优化答案正确性、程序一致性与结构约束
- **benchmark 评估**：`FinQA / ConvFinQA + CFLUE + FinanceBench + AdaptLLM/finance-tasks`

本 Notebook 复用 MedicalGPT 的训练框架：
- 数据格式遵循 `docs/datasets.md`
- pipeline 参考 `run_training_dpo_pipeline.ipynb`


## 环境准备

如果你在全新环境运行，请先安装依赖；已按项目 README 配好可跳过。


In [ ]:
import torch

# 查看 PyTorch 版本
print("PyTorch 版本:", torch.__version__)

# 查看 CUDA 版本（当前PyTorch使用的CUDA）
print("CUDA 版本:", torch.version.cuda)

# 查看是否启用GPU
print("CUDA 是否可用:", torch.cuda.is_available())


In [ ]:
%pip install -r requirements.txt --upgrade


In [ ]:
%pip install modelscope


In [ ]:
# HF Mirror（可选）
import os
HF_ENDPOINT = "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
print("HF_ENDPOINT:", os.getenv("HF_ENDPOINT"))


## 任务配置

这里把配置拆成五部分：
1. 阶段一数据 `SFT-1 (FinQA)`
2. 阶段二数据 `SFT-2 (ConvFinQA)`
3. 原始数据下载缓存目录
4. 转换 / 清洗 / 混合目录
5. 训练与评估输出目录


In [1]:
from pathlib import Path
import json
import random
from itertools import islice

BASE_MODEL = "/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct"

TEMPLATE_NAME = "qwen"
RANDOM_SEED = 42
FORCE_REDOWNLOAD_RAW = False

# ==== 数据处理主链配置 ====
# 本轮不切新根目录，仍沿用 financial_reasoning_v2；只更新 prompt / target / eval 语义。
SFT_VARIANT = "dual_answer_sft"
STRICT_TIERS = "A"
CONVFINQA_MODE = "turn_level"
FILTER_CONFLICTING_PROMPTS = True
SFT2_CONVFINQA_TO_FINQA_RATIO = 2.0
VALIDATION_CONVFINQA_ROWS = 307
VALIDATION_FINQA_ROWS = 153

# ==== 根路径统一配置（数据盘） ====
DISK_ROOT = Path("/root/autodl-tmp")
DATA_DIR = DISK_ROOT / "data" / "financial_reasoning_v2"
OUTPUT_ROOT = DISK_ROOT / "outputs" / "financial_reasoning_v2"

RAW_CACHE_DIR = DISK_ROOT / "data" / "financial_reasoning" / "raw"
SFT1_SFT_DIR = DATA_DIR / "sft1_sharegpt"
SFT2_SFT_DIR = DATA_DIR / "sft2_sharegpt"
DPO_DIR = DATA_DIR / "dpo_pairs"
NORMALIZED_DIR = DATA_DIR / "normalized"
ROUTER_AUDIT_DIR = DATA_DIR / "audit"
MIXED_DIR = DATA_DIR / "mixed"
CLEAN_DIR = DATA_DIR / "clean"
REPORT_DIR = DATA_DIR / "reports"
VALIDATION_DIR = DATA_DIR / "validation"

SFT1_MIXED_FILE = MIXED_DIR / "train_sft1_mixed.jsonl"
SFT2_MIXED_FILE = MIXED_DIR / "train_sft2_mixed.jsonl"
DPO_MIXED_FILE = DPO_DIR / "train_reasoning_dpo.jsonl"
SFT1_CLEAN_FILE = CLEAN_DIR / "train_sft1_clean.jsonl"
SFT2_CLEAN_FILE = CLEAN_DIR / "train_sft2_clean.jsonl"
SFT1_NORMALIZED_DIR = NORMALIZED_DIR / "sft1"
SFT2_NORMALIZED_DIR = NORMALIZED_DIR / "sft2"
SFT1_ROUTER_AUDIT_DIR = ROUTER_AUDIT_DIR / "sft1"
SFT2_ROUTER_AUDIT_DIR = ROUTER_AUDIT_DIR / "sft2"

SFT1_DIR = CLEAN_DIR / "sft1_dir_dual"
SFT2_DIR = CLEAN_DIR / "sft2_dir_dual"
VALIDATION_TRAIN_DIR = VALIDATION_DIR / "train_dir_dual"
DPO_TRAIN_DIR = DPO_DIR / "train_dir"

SFT1_AUDIT_DIR = CLEAN_DIR / "audit_sft1"
SFT2_AUDIT_DIR = CLEAN_DIR / "audit_sft2"
SFT1_STRICT_FILE = CLEAN_DIR / "train_sft1_dual_strict.jsonl"
SFT2_STRICT_FILE = CLEAN_DIR / "train_sft2_dual_balanced.jsonl"
SFT_VALID_FILE = VALIDATION_DIR / "valid_dual_balanced.jsonl"
SFT2_BALANCED_SUMMARY_FILE = CLEAN_DIR / "train_sft2_dual_balanced_summary.json"
SFT2_CONVFINQA_ONLY_FILE = CLEAN_DIR / "train_sft2_convfinqa_turn_dual_strict.jsonl"
SFT2_FINQA_REPLAY_FILE = CLEAN_DIR / "train_sft2_finqa_replay_dual.jsonl"

# 训练/日志输出仍落在 v2 根目录，但目录名标识 dual answer。
SFT1_OUT = OUTPUT_ROOT / "sft1_dual"
SFT1_MERGED_OUT = OUTPUT_ROOT / "sft1_dual_merged"
SFT2_OUT = OUTPUT_ROOT / "sft2_dual"
SFT2_MERGED_OUT = OUTPUT_ROOT / "sft2_dual_merged"
DPO_OUT = OUTPUT_ROOT / "dpo"
DPO_MERGED_OUT = OUTPUT_ROOT / "dpo_merged"
TB_LOG_DIR = OUTPUT_ROOT / "tensorboard"

for d in [
    RAW_CACHE_DIR, SFT1_SFT_DIR, SFT2_SFT_DIR, DPO_DIR, NORMALIZED_DIR, ROUTER_AUDIT_DIR,
    SFT1_NORMALIZED_DIR, SFT2_NORMALIZED_DIR, SFT1_ROUTER_AUDIT_DIR, SFT2_ROUTER_AUDIT_DIR,
    MIXED_DIR, CLEAN_DIR, REPORT_DIR, VALIDATION_DIR,
    SFT1_DIR, SFT2_DIR, VALIDATION_TRAIN_DIR, DPO_TRAIN_DIR, SFT1_AUDIT_DIR, SFT2_AUDIT_DIR,
    OUTPUT_ROOT, TB_LOG_DIR, SFT1_OUT, SFT1_MERGED_OUT, SFT2_OUT, SFT2_MERGED_OUT, DPO_OUT, DPO_MERGED_OUT,
]:
    d.mkdir(parents=True, exist_ok=True)

SFT1_DATA_SPECS = [
    {
        "name": "finqa_train",
        "family": "finqa",
        "source_type": "local",
        "local_path": RAW_CACHE_DIR / "finqa" / "train.json",
        "split": "train",
        "weight": 1.0,
        "max_rows": None,
        "target_rows": None,
        "sft_variant": SFT_VARIANT,
        "strict_tiers": STRICT_TIERS,
    },
]

SFT2_DATA_SPECS = [
    {
        "name": "convfinqa_train_turn",
        "family": "convfinqa_turn",
        "source_type": "local",
        "local_path": RAW_CACHE_DIR / "convfinqa_turn" / "train_turn.json",
        "fallback_local_path": RAW_CACHE_DIR / "convfinqa" / "train.json",
        "split": "train",
        "weight": 1.0,
        "max_rows": None,
        "target_rows": None,
        "sft_variant": SFT_VARIANT,
        "strict_tiers": STRICT_TIERS,
        "convfinqa_mode": CONVFINQA_MODE,
    },
]

ALL_DATA_SPECS = SFT1_DATA_SPECS + SFT2_DATA_SPECS
print(json.dumps({
    "data_dir": str(DATA_DIR),
    "output_root": str(OUTPUT_ROOT),
    "sft_variant": SFT_VARIANT,
    "convfinqa_mode": CONVFINQA_MODE,
    "sft1_file": str(SFT1_STRICT_FILE),
    "sft2_file": str(SFT2_STRICT_FILE),
    "validation_file": str(SFT_VALID_FILE),
}, ensure_ascii=False, indent=2))


{
  "data_dir": "/root/autodl-tmp/data/financial_reasoning_v2",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning_v2",
  "sft_variant": "dual_answer_sft",
  "convfinqa_mode": "turn_level",
  "sft1_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft1_dual_strict.jsonl",
  "sft2_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_dual_balanced.jsonl",
  "validation_file": "/root/autodl-tmp/data/financial_reasoning_v2/validation/valid_dual_balanced.jsonl"
}


### 统一任务格式

本次两阶段 SFT 的主训练模板：
- **FinQA**：文本 + 表格 + 问题 -> 当前样本的 `Evidence / Program / Answer`
- **ConvFinQA**：历史状态 + 当前 turn 问题 + 表格/文本上下文 -> 当前 turn 的 `Evidence / Program / Answer`

ConvFinQA 的字段口径必须区分 current turn 与 final QA：
- 当前 turn question：优先 `annotation.cur_dial[-1]`，缺失时 fallback 到 `annotation.dialogue_break[turn_ind]`
- 当前 turn program：优先 `annotation.cur_program`
- 当前 turn answer：优先 `annotation.exe_ans`
- `qa.question / qa.program_re / qa.exe_ans` 是整题最终问题和最终 program，只能进入 raw metadata 或作为缺失字段 fallback，不能作为 turn-level 主监督

MedicalGPT 对数据格式的要求：
- **SFT**：`conversations`
- **DPO**：`question + response_chosen + response_rejected`

> 注：路由器仍支持 `fineval/fiqa_qa` family，但当前 notebook 的 SFT 两阶段不使用它们。


## 数据集处理流程



### 下载原始数据到本地缓存

这一阶段只负责下载 raw 数据，避免每次重跑 notebook 都重复下载。
- `url_json`：直接下载官方 JSON 文件
- `hf`：通过 `datasets.load_dataset` 落地到本地 jsonl
- 若缓存已存在且 `FORCE_REDOWNLOAD_RAW=False`，则直接跳过


In [2]:
import json
from datasets import load_dataset

ALL_DATA_SPECS = SFT1_DATA_SPECS + SFT2_DATA_SPECS


def raw_cache_file(spec: dict) -> Path:
    if spec["source_type"] == "local":
        return Path(spec["local_path"])
    return RAW_CACHE_DIR / spec["family"] / f"{spec['name']}.jsonl"

raw_files = {}
for spec in ALL_DATA_SPECS:
    cache_file = raw_cache_file(spec)
    raw_files[spec["name"]] = cache_file
    cache_file.parent.mkdir(parents=True, exist_ok=True)

    if spec["source_type"] == "local":
        if not cache_file.exists():
            raise FileNotFoundError(f"Missing local raw file: {cache_file}")
        print(f"[use local] {spec['name']} -> {cache_file}")
        continue

    if cache_file.exists() and not FORCE_REDOWNLOAD_RAW:
        print(f"[skip] use cached raw file: {cache_file}")
        continue

    print(f"[download] {spec['name']} -> {cache_file}")
    if spec["source_type"] == "hf":
        ds = load_dataset(spec["dataset_name"], split=spec["split"])
        with cache_file.open('w', encoding='utf-8') as f:
            for row in ds:
                f.write(json.dumps(dict(row), ensure_ascii=False) + "\n")
    else:
        raise ValueError(f"Unsupported source_type: {spec['source_type']}")
    print(f"[saved] {cache_file}")


[use local] finqa_train -> /root/autodl-tmp/data/financial_reasoning/raw/finqa/train.json
[use local] convfinqa_train_turn -> /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/train_turn.json


### 格式匹配

这一阶段只读取本地 raw 文件，并通过包入口执行当前数据处理主链：
- `python -m financial_data_processors --task sft`：生成 strict SFT，同时可选导出 `normalized / audit`
- `python -m financial_data_processors --task dpo`：基于 strict target 生成轻量 DPO pair

当前 SFT target 固定为英文三段式：`Evidence / Program / Answer`。


In [3]:

import json
import subprocess
import pandas as pd

sft_reports = []
dpo_reports = []

for spec in ALL_DATA_SPECS:
    raw_file = raw_files[spec['name']]
    is_sft1 = spec in SFT1_DATA_SPECS
    out_dir = SFT1_SFT_DIR if is_sft1 else SFT2_SFT_DIR
    norm_dir = SFT1_NORMALIZED_DIR if is_sft1 else SFT2_NORMALIZED_DIR
    audit_dir = SFT1_ROUTER_AUDIT_DIR if is_sft1 else SFT2_ROUTER_AUDIT_DIR

    sft_file = out_dir / f"{spec['name']}_sharegpt.jsonl"
    normalized_file = norm_dir / f"{spec['name']}_normalized.jsonl"
    audit_file = audit_dir / f"{spec['name']}_audit.jsonl"
    dpo_file = DPO_DIR / f"{spec['name']}_dpo.jsonl"

    common_router_args = [
        '--source_file', str(raw_file),
        '--dataset_family', spec['family'],
        '--sft_variant', spec.get('sft_variant', SFT_VARIANT),
        '--strict_tiers', spec.get('strict_tiers', STRICT_TIERS),
        '--filter_conflicting_prompts', 'true' if FILTER_CONFLICTING_PROMPTS else 'false',
    ]
    if spec['family'] == 'convfinqa_turn':
        common_router_args.extend(['--convfinqa_mode', spec.get('convfinqa_mode', CONVFINQA_MODE)])
    sharegpt_cmd = [
        'python', '-m', 'financial_data_processors',
        '--task', 'sft',
        '--output_file', str(sft_file),
        '--normalized_output_file', str(normalized_file),
        '--audit_output_file', str(audit_file),
        *common_router_args,
    ]
    dpo_cmd = [
        'python', '-m', 'financial_data_processors',
        '--task', 'dpo',
        '--output_file', str(dpo_file),
        '--seed', str(RANDOM_SEED),
        *common_router_args,
    ]

    print(' '.join(sharegpt_cmd))
    sharegpt_result = subprocess.run(sharegpt_cmd, check=True, capture_output=True, text=True)
    print(sharegpt_result.stdout)
    sft_reports.append(json.loads(sharegpt_result.stdout))

    print(' '.join(dpo_cmd))
    dpo_result = subprocess.run(dpo_cmd, check=True, capture_output=True, text=True)
    print(dpo_result.stdout)
    dpo_reports.append(json.loads(dpo_result.stdout))

print('[SFT conversion reports]')
print(json.dumps(sft_reports, ensure_ascii=False, indent=2))
print('[DPO conversion reports]')
print(json.dumps(dpo_reports, ensure_ascii=False, indent=2))

sft_summary_rows = []
for r in sft_reports:
    for fam, stats in r.get('per_family', {}).items():
        sft_summary_rows.append({
            'dataset_family': r.get('dataset_family', ''),
            'family': fam,
            'sft_variant': r.get('sft_variant'),
            'strict_tiers': r.get('strict_tiers'),
            'input_rows': stats.get('input_rows', 0),
            'strict_saved_rows': stats.get('saved_rows', 0),
            'normalized_rows': stats.get('normalized_rows', 0),
            'audit_rows': stats.get('audit_rows', 0),
            'tier_A_rows': stats.get('tier_A_rows', 0),
            'tier_B_rows': stats.get('tier_B_rows', 0),
            'tier_C_rows': stats.get('tier_C_rows', 0),
            'requires_history_rows': stats.get('requires_history_rows', 0),
            'multiturn_history_rows': stats.get('multiturn_history_rows', 0),
            'history_answer_missing_rows': stats.get('history_answer_missing_rows', 0),
            'history_full_reasoning_rows': stats.get('history_full_reasoning_rows', 0),
            'history_question_only_rows': stats.get('history_question_only_rows', 0),
            'history_full_reasoning_turns': stats.get('history_full_reasoning_turns', 0),
            'history_question_only_turns': stats.get('history_question_only_turns', 0),
            'rendered_history_full_reasoning_turns': stats.get('rendered_history_full_reasoning_turns', 0),
            'rendered_history_question_only_turns': stats.get('rendered_history_question_only_turns', 0),
            'duplicate_current_question_in_history_source_rows': stats.get('duplicate_current_question_in_history_source_rows', 0),
            'current_answer_leaked_in_history_source_rows': stats.get('current_answer_leaked_in_history_source_rows', 0),
            'question_semantic_risk_rows': stats.get('question_semantic_risk_rows', 0),
            'question_text_suspicious_rows': stats.get('question_text_suspicious_rows', 0),
            'weak_table_evidence_rendering_rows': stats.get('weak_table_evidence_rendering_rows', 0),
            'evidence_not_in_rendered_prompt_rows': stats.get('evidence_not_in_rendered_prompt_rows', 0),
            'evidence_visible_in_prompt_rows': stats.get('evidence_visible_in_prompt_rows', 0),
            'duplicate_current_question_in_history_rows': stats.get('duplicate_current_question_in_history_rows', 0),
            'current_answer_leaked_in_history_rows': stats.get('current_answer_leaked_in_history_rows', 0),
            'table_evidence_column_pruned_rows': stats.get('table_evidence_column_pruned_rows', 0),
            'exact_evidence_alignment_rows': stats.get('exact_evidence_alignment_rows', 0),
            'program_answer_match_rows': stats.get('program_answer_match_rows', 0),
            'raw_program_unchanged_rows': stats.get('raw_program_unchanged_rows', 0),
            'json_like_evidence_rows': stats.get('json_like_evidence_rows', 0),
        })

sft_summary_df = pd.DataFrame(sft_summary_rows)
print()
print('[SFT strict/normalized/audit summary table]')
display(sft_summary_df)

dpo_summary_rows = []
for r in dpo_reports:
    pf = r.get('dpo_post_filter', {})
    dpo_summary_rows.append({
        'dataset_family': r.get('dataset_family', ''),
        'input_rows': r.get('input_rows', 0),
        'saved_rows': r.get('saved_rows', 0),
        'skipped_rows': r.get('skipped_rows', 0),
        'post_filter_input_rows': pf.get('post_filter_input_rows', 0),
        'post_filter_saved_rows': pf.get('post_filter_saved_rows', 0),
        'post_filter_skipped_rows': pf.get('post_filter_skipped_rows', 0),
        'post_filter_duplicate_pair_rows': pf.get('post_filter_duplicate_pair_rows', 0),
        'post_filter_not_comparable_rows': pf.get('post_filter_not_comparable_rows', 0),
        'post_filter_rejected_too_short_rows': pf.get('post_filter_rejected_too_short_rows', 0),
    })

dpo_summary_df = pd.DataFrame(dpo_summary_rows)
print()
print('[DPO post-filter summary table]')
display(dpo_summary_df)


python -m financial_data_processors --task sft --output_file /root/autodl-tmp/data/financial_reasoning_v2/sft1_sharegpt/finqa_train_sharegpt.jsonl --normalized_output_file /root/autodl-tmp/data/financial_reasoning_v2/normalized/sft1/finqa_train_normalized.jsonl --audit_output_file /root/autodl-tmp/data/financial_reasoning_v2/audit/sft1/finqa_train_audit.jsonl --source_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/train.json --dataset_family finqa --sft_variant dual_answer_sft --strict_tiers A --filter_conflicting_prompts true
{
  "task": "sft",
  "output_file": "/root/autodl-tmp/data/financial_reasoning_v2/sft1_sharegpt/finqa_train_sharegpt.jsonl",
  "normalized_output_file": "/root/autodl-tmp/data/financial_reasoning_v2/normalized/sft1/finqa_train_normalized.jsonl",
  "audit_output_file": "/root/autodl-tmp/data/financial_reasoning_v2/audit/sft1/finqa_train_audit.jsonl",
  "dataset_family": "finqa",
  "sft_variant": "dual_answer_sft",
  "strict_tiers": "A",
  "input_rows": 6

,dataset_family,family,sft_variant,strict_tiers,input_rows,strict_saved_rows,normalized_rows,audit_rows,tier_A_rows,tier_B_rows,...,weak_table_evidence_rendering_rows,evidence_not_in_rendered_prompt_rows,evidence_visible_in_prompt_rows,duplicate_current_question_in_history_rows,current_answer_leaked_in_history_rows,table_evidence_column_pruned_rows,exact_evidence_alignment_rows,program_answer_match_rows,raw_program_unchanged_rows,json_like_evidence_rows
0,finqa,finqa,dual_answer_sft,A,6251,3694,6251,2557,3694,746,...,0,1395,4856,0,0,1773,6251,5703,6251,0
1,convfinqa_turn,convfinqa_turn,dual_answer_sft,A,11104,6147,11104,4957,6147,2118,...,0,2432,8672,16,0,2393,11104,10825,11104,0



[DPO post-filter summary table]


,dataset_family,input_rows,saved_rows,skipped_rows,post_filter_input_rows,post_filter_saved_rows,post_filter_skipped_rows,post_filter_duplicate_pair_rows,post_filter_not_comparable_rows,post_filter_rejected_too_short_rows
0,finqa,6251,3667,2584,3694,3667,27,27,0,0
1,convfinqa_turn,11104,6133,4971,6147,6133,14,14,0,0


展示转换后的 `ConvFinQA / FinQA` SFT 样例，确认模板、表格上下文、历史对话和推理程序是否正常。

关键检查：

1. `FinQA` 是单轮表文推理：prompt 中使用 `Question:`，target 固定为 `Evidence / Program / Answer`。
2. `ConvFinQA` 是 turn-level multiturn：prompt 中使用 `Current question:`，该问题必须来自当前 turn，而不是 `qa.question` 的整题最终问题。
3. ConvFinQA history 优先渲染前序完整推理块：`Q + Evidence + Program + Answer`。拿不到可靠前序监督时才使用 question-only fallback。

ConvFinQA 正确样例形态：

```text
Conversation history:
Q: what is the net cash from operating activities in 2009?
Evidence:
- year ended june 30 , cash provided by operations increased $ 25587 to $ 206588 ...

Program: 206588
Answer: 206588

Current question: what about in 2008?

Respond exactly in this format:
Evidence:
- ...

Program: ...
Answer: ...
```

对应 target 应该是当前 turn，而不是最终 percentage change：

```text
Evidence:
- year ended june 30 , cash provided by operations increased $ 25587 to $ 206588 ...

Program: 181001
Answer: 181001
```

metadata 中应同时可见：
- `raw_metadata.current_question / current_program / current_exe_ans`
- `raw_metadata.final_question / final_program_re / final_exe_ans`

如果 `Current question` 等于 `final_question`，而当前 turn 不是最后一轮，说明数据处理又退化成 final-QA 监督，需要停止训练并修复。


### Strict 数据混合

`financial_data_processors` 已在转换阶段完成：
- `program_re` 执行校验
- evidence exact 对齐
- `answer_norm` 选择
- strict / normalized / audit 分流

因此这里不再执行旧的 `clean -> audit_sharegpt -> filter_sharegpt_by_audit` 主过滤流程，而是直接对 router 产出的 strict SFT 文件做抽样与合并，生成训练目录需要的 jsonl。


In [4]:
def sample_jsonl_records(path: Path, target_rows: int | None = None, seed: int = 42):
    with path.open('r', encoding='utf-8') as f:
        records = [line for line in f if line.strip()]
    if target_rows is None or target_rows <= 0 or len(records) <= target_rows:
        return records
    rng = random.Random(seed)
    idxs = list(range(len(records)))
    rng.shuffle(idxs)
    idxs = sorted(idxs[:target_rows])
    return [records[i] for i in idxs]


def sample_records(records, target_rows: int | None = None, seed: int = 42):
    records = list(records)
    if target_rows is None or target_rows <= 0 or len(records) <= target_rows:
        return records
    rng = random.Random(seed)
    idxs = list(range(len(records)))
    rng.shuffle(idxs)
    idxs = sorted(idxs[:target_rows])
    return [records[i] for i in idxs]


def read_jsonl_objects(path: Path):
    raw = path.read_text(encoding='utf-8')
    decoder = json.JSONDecoder()
    rows = []
    pos = 0
    bad_chunks = []
    literal_separator_count = 0

    while pos < len(raw):
        while pos < len(raw) and raw[pos].isspace():
            pos += 1
        while raw.startswith('\\n', pos):
            literal_separator_count += 1
            pos += 2
            while pos < len(raw) and raw[pos].isspace():
                pos += 1
        if pos >= len(raw):
            break
        try:
            obj, next_pos = decoder.raw_decode(raw, pos)
        except json.JSONDecodeError as e:
            bad_chunks.append({
                'char_pos': pos,
                'message': str(e),
                'snippet': raw[pos:pos + 160],
            })
            break
        rows.append(obj)
        pos = next_pos

    report = {
        'path': str(path),
        'rows': len(rows),
        'bad_chunks': len(bad_chunks),
        'literal_separator_count': literal_separator_count,
    }
    return rows, report, bad_chunks


def write_jsonl_objects(path: Path, rows):
    with path.open('w', encoding='utf-8') as wf:
        for row in rows:
            wf.write(json.dumps(row, ensure_ascii=False) + '\n')


In [5]:
def _record_prompt(row):
    conv = row.get("conversations") or []
    if conv and isinstance(conv[0], dict):
        return conv[0].get("value") or conv[0].get("content") or ""
    return row.get("prompt", "")


def _record_answer_norm(row):
    meta = row.get("metadata") or {}
    if meta.get("answer_norm") is not None:
        return str(meta.get("answer_norm"))
    conv = row.get("conversations") or []
    target = ""
    if len(conv) > 1 and isinstance(conv[1], dict):
        target = conv[1].get("value") or conv[1].get("content") or ""
    marker = "Normalized Answer:"
    if marker in target:
        return target.split(marker, 1)[1].strip().splitlines()[0].strip()
    marker = "Answer:"
    if marker in target:
        return target.split(marker, 1)[1].strip().splitlines()[0].strip()
    return ""


def filter_conflicting_prompt_labels(rows):
    prompt_to_norm = {}
    kept = []
    conflicts = 0
    for row in rows:
        prompt = _record_prompt(row)
        norm = _record_answer_norm(row)
        old = prompt_to_norm.get(prompt)
        if old is not None and old != norm:
            conflicts += 1
            continue
        prompt_to_norm[prompt] = norm
        kept.append(row)
    return kept, conflicts


# 1) SFT1 (FinQA) strict-A dual-answer 数据
with SFT1_STRICT_FILE.open("w", encoding="utf-8") as wf:
    for spec in SFT1_DATA_SPECS:
        path = SFT1_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
        sampled = sample_jsonl_records(path, spec.get("target_rows"), seed=RANDOM_SEED)
        for line in sampled:
            wf.write(line if line.endswith("\n") else line + "\n")

finqa_rows, _, bad = read_jsonl_objects(SFT1_STRICT_FILE)
if bad:
    raise ValueError(f"Invalid FinQA strict JSONL: {SFT1_STRICT_FILE} -> {bad[0]}")
finqa_rows, finqa_conflicts = filter_conflicting_prompt_labels(finqa_rows)
write_jsonl_objects(SFT1_STRICT_FILE, finqa_rows)
SFT1_MIXED_FILE.write_text(SFT1_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")
SFT1_CLEAN_FILE.write_text(SFT1_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")

# 2) SFT2: ConvFinQA turn-level + FinQA replay, default ConvFinQA:FinQA = 2:1
convfinqa_rows = []
for spec in SFT2_DATA_SPECS:
    path = SFT2_SFT_DIR / f"{spec['name']}_sharegpt.jsonl"
    rows, _, bad = read_jsonl_objects(path)
    if bad:
        raise ValueError(f"Invalid SFT2 source JSONL: {path} -> {bad[0]}")
    convfinqa_rows.extend(rows)

convfinqa_rows, conv_conflicts = filter_conflicting_prompt_labels(convfinqa_rows)
random.Random(RANDOM_SEED).shuffle(convfinqa_rows)
random.Random(RANDOM_SEED + 1).shuffle(finqa_rows)

valid_conv_target = min(VALIDATION_CONVFINQA_ROWS, len(convfinqa_rows))
valid_finqa_target = min(VALIDATION_FINQA_ROWS, len(finqa_rows))
valid_conv = convfinqa_rows[:valid_conv_target]
valid_finqa = finqa_rows[:valid_finqa_target]
train_conv = convfinqa_rows[valid_conv_target:]
train_finqa_source = finqa_rows[valid_finqa_target:] or finqa_rows

finqa_train_target = int(round(len(train_conv) / SFT2_CONVFINQA_TO_FINQA_RATIO)) if SFT2_CONVFINQA_TO_FINQA_RATIO else len(train_finqa_source)
selected_replay = sample_records(train_finqa_source, finqa_train_target, seed=RANDOM_SEED + 2)
replayed_rows = max(0, len(selected_replay) - len(train_finqa_source))

sft2_rows = train_conv + selected_replay
valid_rows = valid_conv + valid_finqa
random.Random(RANDOM_SEED + 3).shuffle(sft2_rows)
random.Random(RANDOM_SEED + 4).shuffle(valid_rows)

write_jsonl_objects(SFT2_CONVFINQA_ONLY_FILE, train_conv)
write_jsonl_objects(SFT2_FINQA_REPLAY_FILE, selected_replay)
write_jsonl_objects(SFT2_STRICT_FILE, sft2_rows)
write_jsonl_objects(SFT_VALID_FILE, valid_rows)

SFT2_MIXED_FILE.write_text(SFT2_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")
SFT2_CLEAN_FILE.write_text(SFT2_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")

# Make train_file_dir-style folders
(SFT1_DIR / SFT1_STRICT_FILE.name).write_text(SFT1_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")
(SFT2_DIR / SFT2_STRICT_FILE.name).write_text(SFT2_STRICT_FILE.read_text(encoding="utf-8"), encoding="utf-8")
(VALIDATION_TRAIN_DIR / SFT_VALID_FILE.name).write_text(SFT_VALID_FILE.read_text(encoding="utf-8"), encoding="utf-8")

summary = {
    "sft_variant": SFT_VARIANT,
    "convfinqa_mode": CONVFINQA_MODE,
    "same_prompt_conflicting_labels": finqa_conflicts + conv_conflicts,
    "sft1_rows": len(finqa_rows),
    "sft2_convfinqa_source_rows": len(convfinqa_rows),
    "sft2_train_convfinqa_rows": len(train_conv),
    "sft2_train_finqa_replay_rows": len(selected_replay),
    "sft2_train_rows": len(sft2_rows),
    "validation_convfinqa_rows": len(valid_conv),
    "validation_finqa_rows": len(valid_finqa),
    "validation_rows": len(valid_rows),
    "convfinqa_to_finqa_ratio": len(train_conv) / max(len(selected_replay), 1),
    "replayed_rows": replayed_rows,
    "sft2_convfinqa_only_file": str(SFT2_CONVFINQA_ONLY_FILE),
    "sft2_finqa_replay_file": str(SFT2_FINQA_REPLAY_FILE),
    "sft2_balanced_file": str(SFT2_STRICT_FILE),
    "validation_file": str(SFT_VALID_FILE),
}
SFT2_BALANCED_SUMMARY_FILE.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "sft_variant": "dual_answer_sft",
  "convfinqa_mode": "turn_level",
  "same_prompt_conflicting_labels": 0,
  "sft1_rows": 3686,
  "sft2_convfinqa_source_rows": 6145,
  "sft2_train_convfinqa_rows": 5838,
  "sft2_train_finqa_replay_rows": 2919,
  "sft2_train_rows": 8757,
  "validation_convfinqa_rows": 307,
  "validation_finqa_rows": 153,
  "validation_rows": 460,
  "convfinqa_to_finqa_ratio": 2.0,
  "replayed_rows": 0,
  "sft2_convfinqa_only_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_convfinqa_turn_dual_strict.jsonl",
  "sft2_finqa_replay_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_finqa_replay_dual.jsonl",
  "sft2_balanced_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_dual_balanced.jsonl",
  "validation_file": "/root/autodl-tmp/data/financial_reasoning_v2/validation/valid_dual_balanced.jsonl"
}


In [6]:
def line_count(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def _spec_report(spec):
    return {
        "name": spec.get("name"),
        "weight": spec.get("weight"),
        "max_rows": spec.get("max_rows"),
        "target_rows": spec.get("target_rows"),
        "sft_variant": spec.get("sft_variant", SFT_VARIANT),
        "strict_tiers": spec.get("strict_tiers", STRICT_TIERS),
        "convfinqa_mode": spec.get("convfinqa_mode"),
        "sampling_mode": "all_rows" if spec.get("target_rows") is None else "sampled",
    }

balanced_summary = {}
if SFT2_BALANCED_SUMMARY_FILE.exists():
    balanced_summary = json.loads(SFT2_BALANCED_SUMMARY_FILE.read_text(encoding="utf-8"))

report = {
    "disk_root": str(DISK_ROOT),
    "data_dir": str(DATA_DIR),
    "output_root": str(OUTPUT_ROOT),
    "sft_variant": SFT_VARIANT,
    "strict_tiers": STRICT_TIERS,
    "convfinqa_mode": CONVFINQA_MODE,
    "filter_conflicting_prompts": FILTER_CONFLICTING_PROMPTS,
    "target_schema": "Evidence / Program / Answer / Normalized Answer",
    "sft2_mixture": {
        "convfinqa_to_finqa_ratio": SFT2_CONVFINQA_TO_FINQA_RATIO,
        "balanced_summary": balanced_summary,
    },
    "sft1_allocations": [_spec_report(spec) for spec in SFT1_DATA_SPECS],
    "sft2_allocations": [_spec_report(spec) for spec in SFT2_DATA_SPECS],
    "sft1_raw_mix_rows": line_count(SFT1_MIXED_FILE),
    "sft1_clean_rows": line_count(SFT1_CLEAN_FILE),
    "sft1_strict_rows": line_count(SFT1_STRICT_FILE),
    "sft2_raw_mix_rows": line_count(SFT2_MIXED_FILE),
    "sft2_clean_rows": line_count(SFT2_CLEAN_FILE),
    "sft2_strict_rows": line_count(SFT2_STRICT_FILE),
    "sft2_convfinqa_only_rows": line_count(SFT2_CONVFINQA_ONLY_FILE),
    "sft2_finqa_replay_rows": line_count(SFT2_FINQA_REPLAY_FILE),
    "validation_rows": line_count(SFT_VALID_FILE),
}
print(json.dumps(report, ensure_ascii=False, indent=2))


{
  "disk_root": "/root/autodl-tmp",
  "data_dir": "/root/autodl-tmp/data/financial_reasoning_v2",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning_v2",
  "sft_variant": "dual_answer_sft",
  "strict_tiers": "A",
  "convfinqa_mode": "turn_level",
  "filter_conflicting_prompts": true,
  "target_schema": "Evidence / Program / Answer / Normalized Answer",
  "sft2_mixture": {
    "convfinqa_to_finqa_ratio": 2.0,
    "balanced_summary": {
      "sft_variant": "dual_answer_sft",
      "convfinqa_mode": "turn_level",
      "same_prompt_conflicting_labels": 0,
      "sft1_rows": 3686,
      "sft2_convfinqa_source_rows": 6145,
      "sft2_train_convfinqa_rows": 5838,
      "sft2_train_finqa_replay_rows": 2919,
      "sft2_train_rows": 8757,
      "validation_convfinqa_rows": 307,
      "validation_finqa_rows": 153,
      "validation_rows": 460,
      "convfinqa_to_finqa_ratio": 2.0,
      "replayed_rows": 0,
      "sft2_convfinqa_only_file": "/root/autodl-tmp/data/financial_reasoni

### 数据集配比说明（SFT v2）

当前策略：
- SFT-1：仅使用 **FinQA**，先学单轮表文混合数值推理
- SFT-2：使用 **ConvFinQA turn-level strict 样本 + FinQA replay**，再学多轮 follow-up 推理并保留单轮能力
- 如果未设置 `target_rows` 或总 budget，则默认使用该数据源转换后的全部 strict 样本
- SFT 联合训练：使用 `sft_dir_strict`，但必须先基于当前 strict 主链重建

本轮重点不是固定数据配比，而是 **target 设计与可验证性**：
- assistant target 固定为英文三段式：`Evidence / Program / Answer`
- evidence 使用 exact 对齐后的原始证据片段
- FinQA 的 `program_re` 来自原始 FinQA program
- ConvFinQA 的 `program_re` 表示当前 turn program，来源应是 `annotation.cur_program`；`qa.program_re` 是最终整题 program，只能保留为 final metadata
- 重新生成 strict 文件后，必须先检查 schema、evidence、program-answer match、current/final question 是否错位，再启动 SFT


In [7]:

def _norm_prompt_question(text):
    return " ".join("".join(ch.lower() if ch.isalnum() else " " for ch in str(text)).split())


def _prompt_current_question(prompt: str) -> str:
    marker = "Current question: "
    if marker not in prompt:
        return ""
    return prompt.split(marker, 1)[1].split("\n\nRespond", 1)[0].strip()


def _prompt_history_questions(prompt: str):
    questions = []
    question_only_mode = "Conversation history questions:" in prompt
    for line in prompt.splitlines():
        if line.startswith("Q: "):
            questions.append(line[3:].strip())
        elif question_only_mode and line.startswith("- "):
            questions.append(line[2:].strip())
    return questions


def audit_sft_jsonl(path: Path):
    rows = 0
    json_like_evidence = 0
    target_schema_ok = 0
    exact_evidence = 0
    program_answer_match = 0
    raw_program_present = 0
    raw_answer_mismatch = 0
    tier_counts = {'A': 0, 'B': 0, 'C': 0}
    requires_history = 0
    history_turn_rows = 0
    history_answer_missing = 0
    history_full_reasoning_rows = 0
    history_question_only_rows = 0
    history_full_reasoning_turns = 0
    history_question_only_turns = 0
    evidence_visible = 0
    evidence_not_in_prompt = 0
    duplicate_current_question = 0
    rendered_duplicate_current_question = 0
    current_answer_leaked = 0
    question_text_suspicious = 0
    table_evidence_column_pruned = 0
    convfinqa_current_question_metadata_mismatch = 0
    convfinqa_final_question_used_as_current = 0
    avg_prompt_chars = 0
    avg_answer_chars = 0
    first_examples = []

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            rows += 1
            conv = row.get('conversations', [])
            prompt = conv[0]['value'] if len(conv) > 0 else ''
            answer = conv[1]['value'] if len(conv) > 1 else ''
            meta = row.get('metadata', {}) if isinstance(row.get('metadata'), dict) else {}
            raw_meta = meta.get('raw_metadata') if isinstance(meta.get('raw_metadata'), dict) else {}
            audit_flags = meta.get('audit_flags') or []
            semantic_flags = meta.get('semantic_audit_flags') or []

            avg_prompt_chars += len(prompt)
            avg_answer_chars += len(answer)
            if '{"text_' in answer or '{"table_' in answer:
                json_like_evidence += 1
            if all(anchor in answer for anchor in ['Evidence:', 'Program:', 'Answer:']):
                target_schema_ok += 1
            if meta.get('evidence_match_type') == 'exact':
                exact_evidence += 1
            if meta.get('answer_matches_program') is True:
                program_answer_match += 1
            if meta.get('program_raw'):
                raw_program_present += 1
            if 'raw_answer_mismatch_with_answer_norm' in audit_flags:
                raw_answer_mismatch += 1
            tier = str(meta.get('quality_tier') or 'A').upper()
            if tier in tier_counts:
                tier_counts[tier] += 1
            if meta.get('requires_history'):
                requires_history += 1
            if meta.get('history_turns', 0):
                history_turn_rows += 1
            if meta.get('history_answer_missing') is True:
                history_answer_missing += 1
            full_reasoning_turns = int(meta.get('history_full_reasoning_turns') or 0)
            question_only_turns = int(meta.get('history_question_only_turns') or 0)
            if full_reasoning_turns:
                history_full_reasoning_rows += 1
                history_full_reasoning_turns += full_reasoning_turns
            if question_only_turns:
                history_question_only_rows += 1
                history_question_only_turns += question_only_turns
            if meta.get('evidence_visible_in_prompt') is True:
                evidence_visible += 1
            if 'evidence_not_in_rendered_prompt' in audit_flags:
                evidence_not_in_prompt += 1
            if 'duplicate_current_question_in_history' in audit_flags:
                duplicate_current_question += 1
            if 'current_answer_leaked_in_history' in audit_flags:
                current_answer_leaked += 1

            prompt_current_question = _prompt_current_question(prompt)
            current_norm = _norm_prompt_question(prompt_current_question)
            if current_norm and any(_norm_prompt_question(q) == current_norm for q in _prompt_history_questions(prompt)):
                rendered_duplicate_current_question += 1
            raw_current_norm = _norm_prompt_question(raw_meta.get('current_question', ''))
            raw_final_norm = _norm_prompt_question(raw_meta.get('final_question', ''))
            if row.get('source_dataset') == 'ConvFinQA' and raw_current_norm and current_norm and current_norm != raw_current_norm:
                convfinqa_current_question_metadata_mismatch += 1
            if row.get('source_dataset') == 'ConvFinQA' and raw_final_norm and current_norm == raw_final_norm and raw_current_norm != raw_final_norm:
                convfinqa_final_question_used_as_current += 1

            if 'question_text_suspicious' in semantic_flags:
                question_text_suspicious += 1
            if meta.get('table_evidence_column_pruned') is True:
                table_evidence_column_pruned += 1
            if len(first_examples) < 2:
                first_examples.append({
                    'record_id': row.get('record_id', ''),
                    'conversation_id': row.get('conversation_id', ''),
                    'answer_preview': answer[:320],
                    'prompt_preview': prompt[:480],
                    'metadata_preview': {
                        'program_raw': meta.get('program_raw', ''),
                        'answer_norm': meta.get('answer_norm', ''),
                        'answer_display': meta.get('answer_display', ''),
                        'quality_tier': meta.get('quality_tier', ''),
                        'requires_history': meta.get('requires_history'),
                        'history_turns': meta.get('history_turns'),
                        'history_full_reasoning_turns': meta.get('history_full_reasoning_turns'),
                        'history_question_only_turns': meta.get('history_question_only_turns'),
                        'history_full_reasoning_ratio': meta.get('history_full_reasoning_ratio'),
                        'history_answer_missing': meta.get('history_answer_missing'),
                        'history_dependency_type': meta.get('history_dependency_type'),
                        'current_question': raw_meta.get('current_question', ''),
                        'current_program': raw_meta.get('current_program', ''),
                        'current_exe_ans': raw_meta.get('current_exe_ans'),
                        'final_question': raw_meta.get('final_question', ''),
                        'final_program_re': raw_meta.get('final_program_re', ''),
                        'final_exe_ans': raw_meta.get('final_exe_ans'),
                        'prompt_current_question': prompt_current_question,
                        'evidence_visible_in_prompt': meta.get('evidence_visible_in_prompt'),
                        'evidence_match_type': meta.get('evidence_match_type', ''),
                        'audit_flags': meta.get('audit_flags', []),
                        'semantic_audit_flags': meta.get('semantic_audit_flags', []),
                    },
                })

    return {
        'path': str(path),
        'rows': rows,
        'target_schema_ratio': round(target_schema_ok / rows, 6) if rows else 0.0,
        'json_like_evidence_ratio': round(json_like_evidence / rows, 6) if rows else 0.0,
        'exact_evidence_alignment_ratio': round(exact_evidence / rows, 6) if rows else 0.0,
        'program_answer_match_ratio': round(program_answer_match / rows, 6) if rows else 0.0,
        'raw_program_present_ratio': round(raw_program_present / rows, 6) if rows else 0.0,
        'raw_answer_mismatch_ratio': round(raw_answer_mismatch / rows, 6) if rows else 0.0,
        'tier_counts': tier_counts,
        'requires_history_ratio': round(requires_history / rows, 6) if rows else 0.0,
        'history_turn_rows_ratio': round(history_turn_rows / rows, 6) if rows else 0.0,
        'history_answer_missing_ratio': round(history_answer_missing / rows, 6) if rows else 0.0,
        'history_full_reasoning_rows_ratio': round(history_full_reasoning_rows / rows, 6) if rows else 0.0,
        'history_question_only_rows_ratio': round(history_question_only_rows / rows, 6) if rows else 0.0,
        'history_full_reasoning_turns': history_full_reasoning_turns,
        'history_question_only_turns': history_question_only_turns,
        'history_full_reasoning_turn_ratio': round(history_full_reasoning_turns / (history_full_reasoning_turns + history_question_only_turns), 6) if (history_full_reasoning_turns + history_question_only_turns) else 0.0,
        'evidence_visible_in_prompt_ratio': round(evidence_visible / rows, 6) if rows else 0.0,
        'evidence_not_in_prompt_ratio': round(evidence_not_in_prompt / rows, 6) if rows else 0.0,
        'duplicate_current_question_in_history_rows': duplicate_current_question,
        'rendered_duplicate_current_question_rows': rendered_duplicate_current_question,
        'current_answer_leaked_in_history_rows': current_answer_leaked,
        'convfinqa_current_question_metadata_mismatch_rows': convfinqa_current_question_metadata_mismatch,
        'convfinqa_final_question_used_as_current_rows': convfinqa_final_question_used_as_current,
        'convfinqa_current_question_metadata_mismatch_ratio': round(convfinqa_current_question_metadata_mismatch / rows, 6) if rows else 0.0,
        'convfinqa_final_question_used_as_current_ratio': round(convfinqa_final_question_used_as_current / rows, 6) if rows else 0.0,
        'question_text_suspicious_ratio': round(question_text_suspicious / rows, 6) if rows else 0.0,
        'table_evidence_column_pruned_rows': table_evidence_column_pruned,
        'avg_prompt_chars': round(avg_prompt_chars / rows, 2) if rows else 0.0,
        'avg_answer_chars': round(avg_answer_chars / rows, 2) if rows else 0.0,
        'examples': first_examples,
    }

audit_report = {
    'sft1_strict': audit_sft_jsonl(SFT1_STRICT_FILE),
    'sft2_strict': audit_sft_jsonl(SFT2_STRICT_FILE),
    'sft2_convfinqa_only': audit_sft_jsonl(SFT2_CONVFINQA_ONLY_FILE),
    'sft2_finqa_replay': audit_sft_jsonl(SFT2_FINQA_REPLAY_FILE),
}
print(json.dumps(audit_report, ensure_ascii=False, indent=2))


{
  "sft1_strict": {
    "path": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft1_dual_strict.jsonl",
    "rows": 3686,
    "target_schema_ratio": 1.0,
    "json_like_evidence_ratio": 0.0,
    "exact_evidence_alignment_ratio": 1.0,
    "program_answer_match_ratio": 1.0,
    "raw_program_present_ratio": 1.0,
    "raw_answer_mismatch_ratio": 0.0,
    "tier_counts": {
      "A": 3686,
      "B": 0,
      "C": 0
    },
    "requires_history_ratio": 0.0,
    "history_turn_rows_ratio": 0.0,
    "history_answer_missing_ratio": 0.0,
    "history_full_reasoning_rows_ratio": 0.0,
    "history_question_only_rows_ratio": 0.0,
    "history_full_reasoning_turns": 0,
    "history_question_only_turns": 0,
    "history_full_reasoning_turn_ratio": 0.0,
    "evidence_visible_in_prompt_ratio": 1.0,
    "evidence_not_in_prompt_ratio": 0.0,
    "duplicate_current_question_in_history_rows": 0,
    "rendered_duplicate_current_question_rows": 0,
    "current_answer_leaked_in_history_rows": 0,
   

### SFT 数据重建建议

当前 notebook 沿用 `financial_reasoning_v2` 根目录，但训练主链已更新为 QA 前置 + 双答案格式。需要重新运行上面的 router 和 strict/balanced 混合单元，生成：

- `train_sft1_dual_strict.jsonl`
- `train_sft2_convfinqa_turn_dual_strict.jsonl`
- `train_sft2_dual_balanced.jsonl`
- `valid_dual_balanced.jsonl`
- `train_sft2_dual_balanced_summary.json`
- `normalized/*.jsonl`
- `audit/*.jsonl`

重点关注：

- target 是否同时包含 `Answer:` 和 `Normalized Answer:`
- `Normalized Answer:` 是否可解析，尤其百分比题是否统一为小数比率
- target 中是否还有 `const_100`、`const_7`、`const_<number>`
- `same_prompt_conflicting_labels` 是否为 `0`
- 当前问题、输出格式和 normalization rule 是否位于 prompt 前 512 tokens
- SFT2 source ratio 是否接近 ConvFinQA:FinQA = 2:1
- ConvFinQA turn 分布、history dependency 分布是否写入 summary
- `evidence_visible_in_prompt_ratio` 应为 `1.0`
- `avg_prompt_chars` / `p95_prompt_chars` 是否可控


In [8]:
sft_manifest = {
    "entrypoint": "python -m financial_data_processors",
    "data_dir": str(DATA_DIR),
    "output_root": str(OUTPUT_ROOT),
    "sft_variant": SFT_VARIANT,
    "strict_tiers": STRICT_TIERS,
    "convfinqa_mode": CONVFINQA_MODE,
    "filter_conflicting_prompts": FILTER_CONFLICTING_PROMPTS,
    "sft1_strict_file": str(SFT1_STRICT_FILE),
    "sft2_strict_file": str(SFT2_STRICT_FILE),
    "sft2_convfinqa_only_file": str(SFT2_CONVFINQA_ONLY_FILE),
    "sft2_finqa_replay_file": str(SFT2_FINQA_REPLAY_FILE),
    "validation_file": str(SFT_VALID_FILE),
    "sft2_balanced_summary_file": str(SFT2_BALANCED_SUMMARY_FILE),
    "sft1_normalized_dir": str(SFT1_NORMALIZED_DIR),
    "sft2_normalized_dir": str(SFT2_NORMALIZED_DIR),
    "sft1_router_audit_dir": str(SFT1_ROUTER_AUDIT_DIR),
    "sft2_router_audit_dir": str(SFT2_ROUTER_AUDIT_DIR),
    "target_schema": "Evidence / Program / Answer / Normalized Answer",
    "prompt_policy": "QA + context + history; current question and output format first; history last.",
    "answer_policy": "Answer is human-readable; Normalized Answer is the strict numeric eval field.",
    "sft2_mode": "convfinqa_mode=turn_level + FinQA replay balanced 2:1",
    "history_policy": "Build history from annotation.cur_dial/dialogue_break; keep history after context.",
    "note": "Do not modify supervised_finetuning truncation; rebuild strict dual files before starting SFT.",
}
print(json.dumps(sft_manifest, ensure_ascii=False, indent=2))


{
  "entrypoint": "python -m financial_data_processors",
  "data_dir": "/root/autodl-tmp/data/financial_reasoning_v2",
  "output_root": "/root/autodl-tmp/outputs/financial_reasoning_v2",
  "sft_variant": "dual_answer_sft",
  "strict_tiers": "A",
  "convfinqa_mode": "turn_level",
  "filter_conflicting_prompts": true,
  "sft1_strict_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft1_dual_strict.jsonl",
  "sft2_strict_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_dual_balanced.jsonl",
  "sft2_convfinqa_only_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_convfinqa_turn_dual_strict.jsonl",
  "sft2_finqa_replay_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_finqa_replay_dual.jsonl",
  "validation_file": "/root/autodl-tmp/data/financial_reasoning_v2/validation/valid_dual_balanced.jsonl",
  "sft2_balanced_summary_file": "/root/autodl-tmp/data/financial_reasoning_v2/clean/train_sft2_dual_balanced_summary.js

## QuickEval: Base


In [ ]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry base=/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --finqa_max_samples 8 \
  --max_new_tokens 256 \
  --temperature 0.0 \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/benchmark_quick_sft1/base


In [ ]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry base=/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/base_passk/

## SFT

### SFT-1：FinQA 主干推理训练

对应当前设置：
1. 阶段一数据集为 `FinQA`
2. 已展示原始样本与转换后样本
3. 已完成清洗与严格过滤
4. 已统一转换为 MedicalGPT SFT 格式
5. 这里给出 `SFT-1` 训练命令


In [ ]:
BASE_MODEL, SFT1_DIR, str(SFT1_OUT)


In [ ]:
sft1_cmd = [
    'python', 'supervised_finetuning.py',
    '--model_name_or_path', BASE_MODEL,
    '--tokenizer_name_or_path', BASE_MODEL,
    '--train_file_dir', str(SFT1_DIR),

    '--validation_split_percentage', '1',
    '--do_eval',
    '--eval_steps', '100',
    '--eval_strategy', 'steps',

    '--do_train',
    '--use_peft',

    '--num_train_epochs', '2',

    '--per_device_train_batch_size', '1',
    '--max_grad_norm', '1.0',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',

    '--warmup_ratio', '0.05',
    '--weight_decay', '0.05',
    '--learning_rate', '1e-5',

    '--logging_steps', '10',
    '--save_steps', '200',

    '--logging_first_step', 'True',
    '--report_to', 'tensorboard',
    '--logging_dir', str(TB_LOG_DIR / 'sft1'),

    '--model_max_length', '1024',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'bfloat16',

    '--device_map', 'auto',
    
    '--output_dir', str(SFT1_OUT),
    '--template_name', TEMPLATE_NAME,
]
print(' '.join(sft1_cmd))


In [ ]:
!python -m training.supervised_finetuning --model_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --train_file_dir /root/autodl-tmp/data/financial_reasoning_v2/clean/sft1_dir_dual \
    --validation_split_percentage 1 \
    --do_eval --eval_steps 100 --eval_strategy steps --per_device_eval_batch_size 1 --prediction_loss_only True --eval_accumulation_steps 1 --max_eval_samples 32 \
    --do_train --use_peft --num_train_epochs 2 --per_device_train_batch_size 1 --max_grad_norm 1.0 --gradient_accumulation_steps 16 --gradient_checkpointing True \
    --warmup_steps 30 --weight_decay 0.05 --learning_rate 5e-6 \
    --logging_steps 10 --save_steps 200 --logging_first_step True --report_to tensorboard \
    --logging_dir /root/autodl-tmp/outputs/financial_reasoning_v2/tensorboard/sft1_dual \
    --model_max_length 1024 --target_modules q_proj,k_proj,v_proj,o_proj \
    --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
    --torch_dtype bfloat16 --bf16 \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/sft1_dual \
    --template_name qwen \
    --preprocessing_num_workers 16


In [ ]:
!python -m tooling.merge_peft_adapter \
    --base_model /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --lora_model /root/autodl-tmp/outputs/financial_reasoning_v2/sft1_dual \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/sft1_dual_merged


### SFT1 QuickEval

基于已经 merge 的 `SFT1_MERGED_OUT` 做快速 benchmark，先回答一个问题：

> SFT-1（FinQA）训练后，模型是否在 FinQA / ConvFinQA quick benchmark 上不低于 base？

这里不再使用旧的 `/root/autodl-tmp/outputs/financial_reasoning/sft_merged`，统一使用 v2 配置里的 `SFT1_MERGED_OUT`。


> 基于 `/root/autodl-tmp/outputs/financial_reasoning_v2/benchmark_quick_sft1`，这次 quickeval 的结论是：SFT1 有明显正向提升，但样本数只有 8 条 FinQA，只能作为 smoke test，不能当最终 benchmark 结论。

- `base`：`/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct`
- `sft1`：`/root/autodl-tmp/outputs/financial_reasoning_v2/sft1_dual_merged`
- 数据：只跑了 `FinQA test`
- 样本数：8
- `ConvFinQA`：未启用
- 输出目录：
  - `/root/autodl-tmp/outputs/financial_reasoning_v2/benchmark_quick_sft1/base`
  - `/root/autodl-tmp/outputs/financial_reasoning_v2/benchmark_quick_sft1/sft1`

| model | task | num_examples | answer_accuracy | numeric_parse_rate | final_answer_coverage | program_section_coverage | avg_prediction_chars |
|---|---:|---:|---:|---:|---:|---:|---:|
| base | finqa_test | 8 | 0.25 | 1.0 | 1.0 | 1.0 | 671.625 |
| sft1 | finqa_test | 8 | 0.50 | 1.0 | 1.0 | 0.875 | 411.0 |

- `answer_accuracy`：`base 25% -> sft1 50%`
- 正确数：`base 2/8`，`sft1 4/8`
- `sft1` 输出更短：平均 `411` 字符 vs base `672` 字符
- `sft1` 更像训练目标：常见格式是 `Evidence / Program / Answer`
- `sft1` 的程序更接近 FinQA 风格，比如 `divide(8.1, 56.0)`，而 base 更倾向写 Python 代码或自然语言解释

**逐样本变化**
- 两者都对：
  - `ETR/2016/page_23.pdf-2`
  - `FIS/2010/page_70.pdf-2`
- `sft1` 修正了 base 的错误：
  - `INTC/2015/page_41.pdf-4`：base 输出 `14.43%`，sft1 输出 `0.1446428571428571`，匹配 gold `0.14464`
  - `AES/2010/page_227.pdf-3`：base 输出 `87.3%`，sft1 输出 `0.1004`，匹配 gold `0.10039`
- 两者仍错：
  - `ADI/2011/page_61.pdf-2`
  - `MAS/2017/page_27.pdf-2`
  - `SYY/2006/page_71.pdf-1`
  - `GS/2015/page_188.pdf-2`

**判断**
SFT1 是有效的：在这个 quick smoke test 上，FinQA 数值答案准确率翻倍，而且输出更短、更结构化、更贴近 FinQA program 形式。

但还不能下“大幅优于 base”的正式结论，因为只有 8 条样本。下一步建议把 FinQA quickeval 扩到 `32` 或 `64` 条，只跑 `sft1`，确认提升是否稳定；等显存环境稳定或装好 `bitsandbytes` 后，再做 `base vs sft1` 的完整对比。


对比两种sft benchmark_summary: 
- `autodl-tmp/outputs/financial_reasoning_v2/benchmark_quick_sft1/sft1/benchmark_summary.csv`
- `autodl-tmp/outputs/financial_reasoning_v2/benchmark_quick_sft1_v2/benchmark_summary.csv`

差异不大



In [ ]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry sft1=/root/autodl-tmp/outputs/financial_reasoning_v2/sft1_dual_merged \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/sft1_dual_passk/

### SFT-2：ConvFinQA 对话推理强化

在 `SFT-1` 完成后：
- 先 merge `SFT-1 LoRA`
- 再用 `ConvFinQA` 做二阶段 SFT（强化多轮 follow-up 推理）


In [ ]:
str(SFT2_DIR), str(SFT2_OUT), str(TB_LOG_DIR / 'sft2')


In [ ]:
# sft1
!python -m training.supervised_finetuning --model_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --train_file_dir /root/autodl-tmp/data/financial_reasoning_v2/clean/sft1_dir_dual \
    --validation_split_percentage 1 \
    --do_eval --eval_steps 100 --eval_strategy steps --per_device_eval_batch_size 1 --prediction_loss_only True --eval_accumulation_steps 1 --max_eval_samples 32 \
    --do_train --use_peft --num_train_epochs 2 --per_device_train_batch_size 1 --max_grad_norm 1.0 --gradient_accumulation_steps 16 --gradient_checkpointing True \
    --warmup_steps 30 --weight_decay 0.05 --learning_rate 5e-6 \
    --logging_steps 10 --save_steps 200 --logging_first_step True --report_to tensorboard \
    --logging_dir /root/autodl-tmp/outputs/financial_reasoning_v2/tensorboard/sft1_dual \
    --model_max_length 1024 --target_modules q_proj,k_proj,v_proj,o_proj \
    --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
    --torch_dtype bfloat16 --bf16 \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/sft1_dual \
    --template_name qwen \
    --preprocessing_num_workers 16


In [ ]:
# sft2
!python -m training.supervised_finetuning \
    --model_name_or_path /root/autodl-tmp/outputs/financial_reasoning_v2/sft1_dual_merged \
    --tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --train_file_dir /root/autodl-tmp/data/financial_reasoning_v2/clean/sft2_dir_dual \
    --validation_split_percentage 1 --do_eval --eval_steps 100 --eval_strategy steps \
    --per_device_eval_batch_size 1 --prediction_loss_only True --eval_accumulation_steps 1 --max_eval_samples 32 \
    --do_train --use_peft \
    --num_train_epochs 2 --per_device_train_batch_size 1 \
    --max_grad_norm 0.5 --gradient_accumulation_steps 16 --gradient_checkpointing True \
    --learning_rate 5e-6 --warmup_steps 50 --weight_decay 0.05 \
    --logging_steps 10 --save_steps 200 --save_total_limit 2 \
    --logging_first_step True --report_to tensorboard \
    --logging_dir /root/autodl-tmp/outputs/financial_reasoning_v2/tensorboard/sft2_dual \
    --model_max_length 1024 --target_modules q_proj,k_proj,v_proj,o_proj \
    --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
    --torch_dtype bfloat16 --bf16 \
    --device_map auto \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual \
    --preprocessing_num_workers 16 \
    --template_name qwen


In [ ]:
!python -m tooling.merge_peft_adapter \
    --base_model /root/autodl-tmp/outputs/financial_reasoning_v2/sft1_dual_merged \
    --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
    --lora_model /root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual \
    --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged


### v1: SFT-1 + SFT-2 Loss 解读

之前 SFT-2 loss 从 `0.29` 到 `0.3` 附近变化不大，不能只看 loss 判断是否有效。现在尤其要先确认 ConvFinQA 是否真的提供了新的 turn-level 监督。

当前已经修正的数据口径：
- `qa.question / qa.program_re / qa.exe_ans` 是整题最终问题和最终 program
- 当前 turn 必须来自 `annotation.cur_dial[-1] / annotation.cur_program / annotation.exe_ans`
- history 优先使用 `Q + Evidence + Program + Answer`，不能伪造 Q/A

重新训练前先看 audit：
- `convfinqa_final_question_used_as_current_rows = 0`
- `convfinqa_current_question_metadata_mismatch_rows = 0`
- `history_full_reasoning_turn_ratio` 有实际覆盖
- `program_answer_match_ratio` 在 strict 数据中保持高位

修正后，SFT-2 loss 仍可能不大幅下降，原因是 target 仍是同一套 `Evidence / Program / Answer` 格式；真正的收益应该体现在 ConvFinQA turn-level benchmark、follow-up 问题的 program accuracy、以及 history 依赖样本的 answer accuracy 上，而不是只看训练 loss。


### QuickEval

基于已经 merge 的 `SFT2_MERGED_OUT` 做快速 benchmark

做pass@k & pass@1


In [ ]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry base=/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct\
  --model_entry sft=/root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --finqa_max_samples 16 \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --convfinqa_max_samples 16\
  --max_new_tokens 1024 \
  --temperature 0.0 \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/benchmark_quick_dual


In [ ]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry sft2=/root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged\
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/sft2_merged_passk/


对比完了。三个 run 的设置一致：都是 `num_examples=16`，ConvFinQA 8 条 + FinQA 8 条，`pass_k=[1,4,8]`，每题采样 8 次，`temperature=0.7/top_p=0.95/seed=42`。所以可以直接横向比。

**总览**

| model/run | answer acc | pass@1 greedy | pass@1 sampled | program acc | pass@4 | pass@8 | avg chars |
|---|---:|---:|---:|---:|---:|---:|---:|
| base | 62.5% | 62.5% | 81.25% | 6.25% | 81.25% | 81.25% | 539.8 |
| sft1_dual | 25.0% | 25.0% | 37.5% | 37.5% | 56.25% | 56.25% | 234.1 |
| sft2_merged | 81.25% | 81.25% | 81.25% | 81.25% | 93.75% | 93.75% | 268.3 |

结论很清楚：`sft2_merged_passk` 最好，macro answer accuracy 比 base 高 `+18.75pp`，比 `sft1_dual` 高 `+56.25pp`。`sft1_dual` 的答案准确率明显退化，但 program accuracy 比 base 高，说明它更会输出程序格式，却没把最终答案做好。

**分任务**

| task | base acc | sft1_dual acc | sft2_merged acc | sft2 vs base |
|---|---:|---:|---:|---:|
| convfinqa_test | 75.0% | 37.5% | 87.5% | +12.5pp |
| finqa_test | 50.0% | 12.5% | 75.0% | +25.0pp |

`FinQA` 上提升更大：base 只有 4/8，sft2 到 6/8。`ConvFinQA` 上 base 已经比较强，sft2 从 6/8 到 7/8。

**Pass@k**

| task | base pass@4/8 | sft1 pass@4/8 | sft2 pass@4/8 |
|---|---:|---:|---:|
| convfinqa_test | 87.5% / 87.5% | 50.0% / 50.0% | 100% / 100% |
| finqa_test | 75.0% / 75.0% | 62.5% / 62.5% | 87.5% / 87.5% |
| macro | 81.25% / 81.25% | 56.25% / 56.25% | 93.75% / 93.75% |

三个 run 都是 `pass@4 == pass@8`，说明多采到 8 次没有比 4 次带来额外命中；如果要省推理成本，后续可以优先看 `pass@4`。

**程序指标**

`base` 的 `program_accuracy` 很低，macro 只有 `6.25%`，但答案准确率还不错，说明它经常用自然语言答对、程序格式不匹配。`sft1_dual` program acc 提到 `37.5%`，但答案掉到 `25%`，像是格式学习压过了数值推理。`sft2_merged` 同时把答案和程序都拉到 `81.25%`，而且 manifest 里标了 `"primary_metric": "executed_answer_accuracy"`，这组结果更符合“可执行 program + 正确答案”的目标。

**逐题正误重叠**

按 greedy 的 16 条题看：

| base, sft1, sft2 正误模式 | 题数 |
|---|---:|
| 三者都对 | 3 |
| base 对、sft1 错、sft2 对 | 6 |
| base 错、sft1 错、sft2 对 | 3 |
| base 错、sft1 对、sft2 对 | 1 |
| base 对、sft1 错、sft2 错 | 1 |
| 三者都错 | 2 |

也就是说，`sft2` 基本保住了 base 的大部分正确题，并额外修复了 4 条 base 错题；只有 1 条是 base 对但 sft2 错。`sft1` 则丢了很多 base 已经会的题。

结果文件位置：

- [base benchmark_summary.csv](/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/base_passk/benchmark_summary.csv)
- [sft1_dual benchmark_summary.csv](/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/sft1_dual_passk/benchmark_summary.csv)
- [sft2_merged benchmark_summary.csv](/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/sft2_merged_passk/benchmark_summary.csv)

一句话：这轮小样本 pass@k 里，`sft2_merged` 是唯一值得继续推进的模型；`sft1_dual` 需要回看训练数据/目标格式，尤其是为什么 program acc 上升但 answer acc 大幅下降。

## DPO

轻量 DPO 目标：
- 优化表达质量
- 保持结构完整
- 减少废话
- 不让偏好训练覆盖主干 reasoning 能力


In [9]:
all_specs = SFT1_DATA_SPECS + SFT2_DATA_SPECS
DPO_TOTAL_BUDGET_VALUE = globals().get('DPO_TOTAL_BUDGET')
MAX_DPO_PER_DATASET_VALUE = globals().get('MAX_DPO_PER_DATASET')
total_source_target = sum((spec.get('target_rows') or 0) for spec in all_specs if (spec.get('target_rows') or 0) > 0)

mixed_rows = []
dpo_mix_report = []

for spec in all_specs:
    path = DPO_DIR / f"{spec['name']}_dpo.jsonl"
    source_rows, source_report, source_bad_chunks = read_jsonl_objects(path)
    if source_bad_chunks:
        raise ValueError(f"Invalid DPO source JSONL: {path} -> {source_bad_chunks[0]}")

    if DPO_TOTAL_BUDGET_VALUE is None:
        dpo_target = spec.get('dpo_target_rows') or spec.get('target_rows') or spec.get('max_rows') or len(source_rows)
    elif total_source_target > 0:
        dpo_target = int(round(DPO_TOTAL_BUDGET_VALUE * (spec.get('target_rows') or 0) / total_source_target))
    else:
        dpo_target = DPO_TOTAL_BUDGET_VALUE

    if MAX_DPO_PER_DATASET_VALUE is not None:
        dpo_target = min(MAX_DPO_PER_DATASET_VALUE, dpo_target)
    if spec.get('max_rows') is not None:
        dpo_target = min(int(spec['max_rows']), dpo_target)

    sampled_rows = sample_records(source_rows, dpo_target, seed=RANDOM_SEED)
    mixed_rows.extend(sampled_rows)
    dpo_mix_report.append({
        'source_dataset': spec['name'],
        'source_rows': len(source_rows),
        'target_rows': dpo_target,
        'sampled_rows': len(sampled_rows),
        'sampling_mode': 'all_rows' if dpo_target is None or dpo_target >= len(source_rows) else 'sampled',
        'literal_separator_count': source_report['literal_separator_count'],
    })

write_jsonl_objects(DPO_MIXED_FILE, mixed_rows)
normalized_rows, normalized_report, normalized_bad_chunks = read_jsonl_objects(DPO_MIXED_FILE)
if normalized_bad_chunks:
    raise ValueError(f"Invalid mixed DPO JSONL after rewrite: {normalized_bad_chunks[0]}")

write_jsonl_objects(DPO_TRAIN_DIR / DPO_MIXED_FILE.name, normalized_rows)

print('[DPO mix normalization report]')
print(json.dumps({
    'dpo_total_budget': DPO_TOTAL_BUDGET_VALUE,
    'max_dpo_per_dataset': MAX_DPO_PER_DATASET_VALUE,
    'mixed_rows': len(normalized_rows),
    'literal_separator_count_in_mixed_file': normalized_report['literal_separator_count'],
    'sources': dpo_mix_report,
}, ensure_ascii=False, indent=2))


[DPO mix normalization report]
{
  "dpo_total_budget": null,
  "max_dpo_per_dataset": null,
  "mixed_rows": 9800,
  "literal_separator_count_in_mixed_file": 0,
  "sources": [
    {
      "source_dataset": "finqa_train",
      "source_rows": 3667,
      "target_rows": 3667,
      "sampled_rows": 3667,
      "sampling_mode": "all_rows",
      "literal_separator_count": 0
    },
    {
      "source_dataset": "convfinqa_train_turn",
      "source_rows": 6133,
      "target_rows": 6133,
      "sampled_rows": 6133,
      "sampling_mode": "all_rows",
      "literal_separator_count": 0
    }
  ]
}


In [ ]:
!python -m training.dpo_training --model_name_or_path /root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged \
--tokenizer_name_or_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
--template_name qwen \
--validation_split_percentage 1 --eval_strategy no \
--train_file_dir /root/autodl-tmp/data/financial_reasoning_v2/dpo_pairs/train_dir \
--do_train --use_peft True \
--per_device_train_batch_size 1 --gradient_accumulation_steps 16 --gradient_checkpointing True \
--learning_rate 5e-6 --max_steps 100 --max_source_length 512 --max_target_length 256 \
--logging_steps 10 --save_steps 40  --logging_first_step True \
--target_modules q_proj,k_proj,v_proj,o_proj \
--lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \
--torch_dtype bfloat16 --device_map auto \
--ddp_find_unused_parameters False \
--output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/dpo


1. 模型有没有学会“偏好 chosen > rejected”
2. 这种偏好学习是健康推进，还是过拟合/塌缩
3. 训练是否已经到该停的时候


最基础的是**判断训练有没有真正发生作用**。

这主要看 `train/loss` 和 `train/rewards/accuracies`：loss 是否持续下降、reward accuracy 是否从较低水平逐步上升。
- 如果这两个指标没有明显变化，说明模型几乎没有学到偏好；而如果 loss 快速下降、accuracy 很快接近 1，则说明模型已经很好地拟合了训练集中的 chosen/rejected 偏好关系，但这时反而需要警惕过拟合，因为 DPO 的目标是泛化偏好，而不是在训练集上“记住答案”。

在确认训练有效之后，第二步要关注模型到底 **“学了什么”** 。

这需要结合 `logps/chosen` 和 `logps/rejected` 来看。
- 理想情况是：模型对 chosen 的概率提高（logps 变得不那么负），同时对 rejected 的概率降低，这意味着模型既在强化正确回答，也在抑制错误回答。
- 但在很多实际训练中（包括你现在的曲线），后期往往表现为 chosen 基本不变，而 rejected 急剧下降，这说明模型主要是在“打压 rejected”，而不是进一步提升正向推理能力。
- 这种现象在对齐阶段非常常见，但对金融数值推理任务来说可能带来风险——模型更像是在模仿 preferred 模板，而不是更好地进行计算和推理。

第三步则是判断训练是否进入 **“过度优化”或“偏好饱和”** 阶段。

这主要依赖 `entropy`、`grad_norm` 和 `reward accuracy` 的联合判断：
1. 当 entropy 持续下降，说明模型输出分布越来越确定、越来越“自信”；
2. grad_norm 明显衰减，说明每一步参数更新的有效信息越来越少'；
3. reward accuracy 已接近 1。

这三者同时出现时，通常意味着模型已经把训练集偏好学完，后续训练收益极小，甚至可能开始损害泛化能力。
- 这也是为什么 DPO 训练往往需要“早停”，而不能单纯依赖 loss 最低来选择模型。

除此之外，还有一些辅助指标帮助你更细致理解训练过程。
- 例如 `mean_token_accuracy` 可以反映模型逐 token 的预测能力，但在 DPO 中它并不是核心指标，因为 DPO 优化的是排序而不是逐 token 拟合；
- `learning_rate` 帮助你解释 loss 或 grad_norm 的变化是否来自学习率调度；
- `epoch` 和 `num_tokens` 则用于判断数据是否被反复使用过多，从而导致过拟合风险。这些指标本身不直接决定模型好坏，但在解释训练行为时非常关键。

总结来说，DPO 曲线的分析核心不是“哪个指标更高或更低”，而是看这些指标之间是否形成一致的逻辑：是否有效学习了偏好、这种学习是正向增强还是负向打压、以及是否已经进入饱和甚至过拟合阶段。最终，任何训练曲线的结论都必须通过下游金融推理 benchmark 来验证，而不能仅凭训练指标判断模型优劣。


| 指标                       | 含义                        | 健康趋势    | 危险信号       | 主要作用       |
| ------------------------ | ------------------------- | ------- | ---------- | ---------- |
| train/loss               | 偏好优化目标（chosen > rejected） | 持续下降后趋稳 | 快速降到极低     | 判断训练是否有效   |
| rewards/accuracies       | chosen 是否优于 rejected 的比例  | 上升并接近高值 | 很快达到 1 并稳定 | 判断偏好是否已学满  |
| logps/chosen             | 对正确回答的概率                  | 逐渐上升    | 基本不变       | 判断正向增强     |
| logps/rejected           | 对错误回答的概率                  | 逐渐下降    | 过度下降       | 判断负向打压     |
| logits/chosen / rejected | 原始打分                      | 两者差距拉大  | 波动异常       | 辅助验证偏好分离   |
| entropy                  | 输出分布不确定性                  | 缓慢下降    | 持续大幅下降     | 判断模型是否变“僵” |
| grad_norm                | 参数更新幅度                    | 逐渐下降    | 过快衰减或爆炸    | 判断是否收敛或不稳定 |
| mean_token_accuracy      | token级预测准确率               | 稳定或略升   | 明显下降或无变化   | 判断是否提升生成能力 |
| learning_rate            | 学习率变化                     | 按计划变化   | 异常波动       | 辅助解释训练变化   |
| epoch                    | 数据遍历次数                    | 平稳增加    | 过高         | 判断是否重复训练过多 |
| num_tokens               | 累计训练token数                | 线性增长    | 无明显异常      | 衡量训练规模     |


> **DPO 的理想状态是：loss 下降、reward accuracy 提升、chosen 上升 + rejected 下降，同时 entropy 和 grad_norm 平稳衰减；一旦 reward accuracy≈1 且 entropy/grad_norm 同时塌缩，就需要考虑早停并转向 benchmark 验证。**


In [ ]:
!python -m evaluation.evaluate_financial_benchmarks \
  --tokenizer_path /root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct \
  --model_entry dpo=/root/autodl-tmp/outputs/financial_reasoning_v2/sft2_dual_merged\
  --adapter_entry dpo=/root/autodl-tmp/outputs/financial_reasoning_v2/dpo \
  --finqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/finqa/test.json \
  --convfinqa_test_file /root/autodl-tmp/data/financial_reasoning/raw/convfinqa_turn/dev_turn.json \
  --finqa_max_samples 8 \
  --convfinqa_max_samples 8 \
  --max_new_tokens 1024 \
  --pass_k 1,4,8 \
  --num_samples_per_example 8 \
  --sample_temperature 0.7 \
  --sample_top_p 0.95 \
  --sample_seed 42 \
  --output_dir /root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/dpo_passk


## 评估：SFT vs SFT+DPO

（当前主测 FinQA + ConvFinQA）

按照 `README_fingpt.md` 的 fix 计划，这里比较 `SFT` 和 `DPO`：
- 任务结果：`answer_accuracy`、`program_accuracy`、`numeric_parse_rate`
- 输出质量：`final_answer_coverage`、`program_section_coverage`、`structured_response_coverage`、`avg_prediction_chars`

评测设置：
- `base`：基座模型
- `sft`：`SFT` merge 后模型
- `dpo`：以 `SFT` merge 模型为 base，再挂载 `DPO` LoRA adapter
- `ConvFinQA`：优先 `dev_turn.json`
- `FinQA`：优先 `test.json`，不存在则回退 `dev.json`


**加入 DPO 后总览**

| run | answer acc | pass@1 greedy | pass@1 sampled | program acc | pass@4 | pass@8 | avg chars |
|---|---:|---:|---:|---:|---:|---:|---:|
| base | 62.5% | 62.5% | 81.25% | 6.25% | 81.25% | 81.25% | 539.8 |
| sft1_dual | 25.0% | 25.0% | 37.5% | 37.5% | 56.25% | 56.25% | 234.1 |
| sft2_merged | 81.25% | 81.25% | 81.25% | 81.25% | 93.75% | 93.75% | 268.3 |
| dpo | 81.25% | 81.25% | 75.0% | 81.25% | 93.75% | 93.75% | 269.0 |

**分任务**

| task | sft2 answer acc | dpo answer acc | sft2 pass@1 sampled | dpo pass@1 sampled |
|---|---:|---:|---:|---:|
| convfinqa_test | 87.5% | 87.5% | 100% | 87.5% |
| finqa_test | 75.0% | 75.0% | 62.5% | 62.5% |
| macro | 81.25% | 81.25% | 81.25% | 75.0% |

结论：**DPO 没有提升 greedy / pass@k 主指标，和 sft2 基本打平；sampled pass@1 还略降了 6.25pp。**

我还对齐了 `sft2_predictions.jsonl` 和 `dpo_predictions.jsonl` 的 greedy 明细：16 条里没有正误翻转，DPO 和 sft2 的每题 answer correctness 完全一样。只有 4 条 FinQA 的输出文本有轻微格式/空格/证据串差异，但正误和 program correctness 不变。

sampled 明细里 DPO 略差一些：

| record | sft2 sampled correct | dpo sampled correct |
|---|---:|---:|
| `convfinqa_test / Double_RSG/2016/page_144.pdf_1` | 3/8 | 2/8 |
| `finqa_test / INTC/2015/page_41.pdf-4` | 7/8 | 5/8 |
| `finqa_test / STT/2007/page_111.pdf-3` | 7/8 | 6/8 |

所以目前这组小样本结果下，我会把排序定为：

`dpo ~= sft2_merged > base >> sft1_dual`

但如果只看是否值得保留 DPO：**当前 DPO adapter 没带来可见收益，甚至让采样稳定性稍微变差。**更像是“没有破坏 greedy 主结果”，而不是“进一步优化了 sft2”。

结果文件在这里：

- [dpo benchmark_summary.csv](/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/dpo_passk/benchmark_summary.csv)
- [dpo benchmark_manifest.json](/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/dpo_passk/benchmark_manifest.json)
- [dpo predictions](/root/autodl-tmp/outputs/financial_reasoning_v2/benchmarks/dpo_passk/dpo_predictions.jsonl)